In [2]:
import pandas as pd
import numpy as np

from pymongo import MongoClient
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["aqi_predictor"]

collection = db["aqi_features"]

df = pd.DataFrame(list(collection.find()))

In [4]:
if "_id" in df.columns:
    df.drop(columns="_id", inplace=True)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp")

df.reset_index(drop=True, inplace=True)

print(df.shape)

df.head()

(8496, 18)


,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed
0,karachi,2025-08-05 08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,44.97,0.05,0.31,76.93,29.8,70,1002.5,20.9
1,karachi,2025-08-05 09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,44.90,0.05,0.31,76.80,29.5,71,1002.2,20.0
2,karachi,2025-08-05 10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,44.71,0.05,0.31,76.69,29.3,72,1002.0,19.5
3,karachi,2025-08-05 11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,44.39,0.06,0.31,77.06,29.2,72,1001.6,19.9
4,karachi,2025-08-05 12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,43.94,0.06,0.31,77.63,28.9,73,1001.4,19.5


In [5]:
# AQI lag features
df["aqi_lag_1"] = df["aqi"].shift(1)
df["aqi_lag_3"] = df["aqi"].shift(3)
df["aqi_lag_6"] = df["aqi"].shift(6)
df["aqi_lag_12"] = df["aqi"].shift(12)
df["aqi_lag_24"] = df["aqi"].shift(24)

# PM2.5 lag features
df["pm25_lag_1"] = df["pm25"].shift(1)
df["pm25_lag_6"] = df["pm25"].shift(6)
df["pm25_lag_24"] = df["pm25"].shift(24)

df.head(30)

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,...,pressure,wind_speed,aqi_lag_1,aqi_lag_3,aqi_lag_6,aqi_lag_12,aqi_lag_24,pm25_lag_1,pm25_lag_6,pm25_lag_24
0,karachi,2025-08-05 08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,...,1002.5,20.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,karachi,2025-08-05 09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,...,1002.2,20.0,48.0,NaN,NaN,NaN,NaN,11.49,NaN,NaN
2,karachi,2025-08-05 10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,...,1002.0,19.5,47.0,NaN,NaN,NaN,NaN,11.23,NaN,NaN
3,karachi,2025-08-05 11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,...,1001.6,19.9,46.0,48.0,NaN,NaN,NaN,10.95,NaN,NaN
4,karachi,2025-08-05 12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,...,1001.4,19.5,44.0,47.0,NaN,NaN,NaN,10.50,NaN,NaN
5,karachi,2025-08-05 13:00:00+00:00,13,5,8,1,42,-1.0,10.13,42.87,...,1001.6,16.2,43.0,46.0,NaN,NaN,NaN,10.21,NaN,NaN
6,karachi,2025-08-05 14:00:00+00:00,14,5,8,1,43,1.0,10.26,40.95,...,1001.8,14.7,42.0,44.0,48.0,NaN,NaN,10.13,11.49,NaN
7,karachi,2025-08-05 15:00:00+00:00,15,5,8,1,44,1.0,10.64,41.22,...,1002.2,14.5,43.0,43.0,47.0,NaN,NaN,10.26,11.23,NaN
8,karachi,2025-08-05 16:00:00+00:00,16,5,8,1,46,2.0,11.09,44.47,...,1002.8,15.6,44.0,42.0,46.0,NaN,NaN,10.64,10.95,NaN
9,karachi,2025-08-05 17:00:00+00:00,17,5,8,1,48,2.0,11.44,48.09,...,1003.5,14.8,46.0,43.0,44.0,NaN,NaN,11.09,10.50,NaN


In [6]:
# AQI rolling mean
df["aqi_roll_mean_6"] = df["aqi"].rolling(6).mean()

df["aqi_roll_mean_12"] = df["aqi"].rolling(12).mean()

df["aqi_roll_mean_24"] = df["aqi"].rolling(24).mean()

# AQI rolling standard deviation
df["aqi_roll_std_24"] = df["aqi"].rolling(24).std()

df.head(30)

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,...,aqi_lag_6,aqi_lag_12,aqi_lag_24,pm25_lag_1,pm25_lag_6,pm25_lag_24,aqi_roll_mean_6,aqi_roll_mean_12,aqi_roll_mean_24,aqi_roll_std_24
0,karachi,2025-08-05 08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,karachi,2025-08-05 09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,...,NaN,NaN,NaN,11.49,NaN,NaN,NaN,NaN,NaN,NaN
2,karachi,2025-08-05 10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,...,NaN,NaN,NaN,11.23,NaN,NaN,NaN,NaN,NaN,NaN
3,karachi,2025-08-05 11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,...,NaN,NaN,NaN,10.95,NaN,NaN,NaN,NaN,NaN,NaN
4,karachi,2025-08-05 12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,...,NaN,NaN,NaN,10.50,NaN,NaN,NaN,NaN,NaN,NaN
5,karachi,2025-08-05 13:00:00+00:00,13,5,8,1,42,-1.0,10.13,42.87,...,NaN,NaN,NaN,10.21,NaN,NaN,45.000000,NaN,NaN,NaN
6,karachi,2025-08-05 14:00:00+00:00,14,5,8,1,43,1.0,10.26,40.95,...,48.0,NaN,NaN,10.13,11.49,NaN,44.166667,NaN,NaN,NaN
7,karachi,2025-08-05 15:00:00+00:00,15,5,8,1,44,1.0,10.64,41.22,...,47.0,NaN,NaN,10.26,11.23,NaN,43.666667,NaN,NaN,NaN
8,karachi,2025-08-05 16:00:00+00:00,16,5,8,1,46,2.0,11.09,44.47,...,46.0,NaN,NaN,10.64,10.95,NaN,43.666667,NaN,NaN,NaN
9,karachi,2025-08-05 17:00:00+00:00,17,5,8,1,48,2.0,11.44,48.09,...,44.0,NaN,NaN,11.09,10.50,NaN,44.333333,NaN,NaN,NaN


In [7]:
# Day 1 (next 24 hours)
df["target_day1"] = [
    df["aqi"].iloc[i+1:i+25].mean()
    if i + 24 < len(df)
    else np.nan
    for i in range(len(df))
]

# Day 2 (25–48 hours)
df["target_day2"] = [
    df["aqi"].iloc[i+25:i+49].mean()
    if i + 48 < len(df)
    else np.nan
    for i in range(len(df))
]

# Day 3 (49–72 hours)
df["target_day3"] = [
    df["aqi"].iloc[i+49:i+73].mean()
    if i + 72 < len(df)
    else np.nan
    for i in range(len(df))
]

In [8]:
df = df.dropna()

df.reset_index(drop=True, inplace=True)

print(df.shape)

(8400, 33)


In [9]:
FEATURES = [

    # Calendar
    "hour",
    "day",
    "month",
    "day_of_week",

    # Pollutants
    "pm25",
    "pm10",
    "o3",
    "no2",
    "so2",
    "co",

    # Weather
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",

    # AQI lag
    "aqi_lag_1",
    "aqi_lag_3",
    "aqi_lag_6",
    "aqi_lag_12",
    "aqi_lag_24",

    # PM2.5 lag
    "pm25_lag_1",
    "pm25_lag_6",
    "pm25_lag_24",

    # Rolling
    "aqi_roll_mean_6",
    "aqi_roll_mean_12",
    "aqi_roll_mean_24",
    "aqi_roll_std_24",
]

In [10]:
X = df[FEATURES]

y_day1 = df["target_day1"]

y_day2 = df["target_day2"]

y_day3 = df["target_day3"]

print(X.shape)

(8400, 26)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y1_train, y1_test = train_test_split(
    X,
    y_day1,
    test_size=0.2,
    shuffle=False,
)

_, _, y2_train, y2_test = train_test_split(
    X,
    y_day2,
    test_size=0.2,
    shuffle=False,
)

_, _, y3_train, y3_test = train_test_split(
    X,
    y_day3,
    test_size=0.2,
    shuffle=False,
)

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import numpy as np
import pandas as pd

In [13]:
from xgboost import XGBRegressor

In [14]:
xgb_day1 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_day1.fit(X_train, y1_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [15]:
xgb_day2 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_day2.fit(X_train, y2_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [16]:
xgb_day3 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_day3.fit(X_train, y3_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [17]:
pred_day1 = xgb_day1.predict(X_test)

pred_day2 = xgb_day2.predict(X_test)

pred_day3 = xgb_day3.predict(X_test)

In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [22]:
day1_mae = mean_absolute_error(y1_test, pred_day1)
day1_rmse = np.sqrt(mean_squared_error(y1_test, pred_day1))
day1_r2 = r2_score(y1_test, pred_day1)

day2_mae = mean_absolute_error(y2_test, pred_day2)
day2_rmse = np.sqrt(mean_squared_error(y2_test, pred_day2))
day2_r2 = r2_score(y2_test, pred_day2)

day3_mae = mean_absolute_error(y3_test, pred_day3)
day3_rmse = np.sqrt(mean_squared_error(y3_test, pred_day3))
day3_r2 = r2_score(y3_test, pred_day3)

print(f"Day 1 - MAE: {day1_mae:.2f}, RMSE: {day1_rmse:.2f}, R2: {day1_r2:.2f}")
print(f"Day 2 - MAE: {day2_mae:.2f}, RMSE: {day2_rmse:.2f}, R2: {day2_r2:.2f}")
print(f"Day 3 - MAE: {day3_mae:.2f}, RMSE: {day3_rmse:.2f}, R2: {day3_r2:.2f}")

Day 1 - MAE: 6.52, RMSE: 9.85, R2: 0.70
Day 2 - MAE: 10.88, RMSE: 15.49, R2: 0.22
Day 3 - MAE: 12.96, RMSE: 17.89, R2: -0.05


In [23]:
import joblib

joblib.dump(xgb_day1, "../models/xgboost_day1.pkl")

joblib.dump(xgb_day2, "../models/xgboost_day2.pkl")

joblib.dump(xgb_day3, "../models/xgboost_day3.pkl")

print("XGBoost model saved successfully.")

XGBoost model saved successfully.


In [24]:
results = pd.DataFrame({
    "Forecast": [
        "Day 1",
        "Day 2",
        "Day 3",
    ],
    "MAE": [
        day1_mae,
        day2_mae,
        day3_mae,
    ],
    "RMSE": [
        day1_rmse,
        day2_rmse,
        day3_rmse,
    ],
    "R²": [
        day1_r2,
        day2_r2,
        day3_r2,
    ],
})

results

,Forecast,MAE,RMSE,R²
0,Day 1,6.518217,9.853910,0.695927
1,Day 2,10.880656,15.491454,0.221235
2,Day 3,12.963224,17.892137,-0.049703
